# Tests of Significance
**ENGR 1451/2451 · Exploratory Data Science · w09a**

Practice Python functions for tests of a **population mean**, following the lecture's die, polymer-strength, and instrument-calibration examples. Run cells in order, then complete the single exercise at the end. All data are included.

**Real-data default:** the population SD $\sigma$ is usually **unknown**. For a mean, estimate it using the sample SD $s$ and use Student's $t$.

Choose the hypotheses and significance level $\alpha$ **before examining the result**. We use $\alpha=0.05$ throughout.


In [2]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.precision", 6)
alpha = 0.05


## 1. Standardize a discrepancy: `np.sqrt()`

The slides compare **100** and **1,000** die rolls with the same observed mean, $\bar{x}=3.68$. Test $H_0:\mu=3.5$ against $H_A:\mu\ne3.5$.

Under the null, assume independent rolls with each face equally likely. This fair-die model specifies $\sigma_0=\sqrt{35/12}$, so

$$\mathrm{SE}_0(\bar X)=\frac{\sigma_0}{\sqrt n},\qquad z=\frac{\bar x-\mu_0}{\mathrm{SE}_0(\bar X)}.$$

Use a NumPy array to calculate both cases at once. The difference stays the same; the SE changes. The normal reference is an **approximation** to the sampling distribution under the fair-die model.


In [3]:
die_mu0 = 3.5
die_sigma0 = np.sqrt(35 / 12)
die_mean = 3.68
die_n = np.array([100, 1000])

die_se = die_sigma0 / np.sqrt(die_n)
die_z = (die_mean - die_mu0) / die_se

pd.DataFrame({
    "Number of rolls": die_n,
    "Observed mean": die_mean,
    "Difference from null": die_mean - die_mu0,
    "SE under H0": die_se,
    "z": die_z,
})


,Number of rolls,Observed mean,Difference from null,SE under H0,z
0,100,3.68,0.18,0.170783,1.053972
1,1000,3.68,0.18,0.054006,3.332952


## 2. Choose the tail: `model.cdf()` and `model.sf()`

As in the lecture, first create a reference distribution: `normal_model = stats.norm()`.

`cdf(z)` gives the **lower-tail area**; `sf(z)` gives the **upper-tail area**. For a symmetric standard normal reference, `2 * normal_model.sf(abs(z))` gives the two-sided $P$-value. With an array, use `np.abs()`.

The table compares alternatives to demonstrate the functions; the die question uses **two sides**. Choose the alternative from the scientific question before inspecting the result.

For 100 rolls, full-precision Python gives $P\approx0.2919$, versus about $0.294$ when the lecture uses rounded $z=1.05$.


In [4]:
normal_model = stats.norm()
die_p_lower = normal_model.cdf(die_z)
die_p_upper = normal_model.sf(die_z)
die_p_two = 2 * normal_model.sf(np.abs(die_z))

pd.DataFrame({
    "Number of rolls": die_n,
    "z": die_z,
    "P: mean < 3.5": die_p_lower,
    "P: mean > 3.5": die_p_upper,
    "P: mean != 3.5": die_p_two,
    "Reject H0 (two-sided, alpha=0.05)": die_p_two <= alpha,
})


,Number of rolls,z,P: mean < 3.5,P: mean > 3.5,P: mean != 3.5,"Reject H0 (two-sided, alpha=0.05)"
0,100,1.053972,0.854052,0.145948,0.291896,False
1,1000,3.332952,0.999570,0.000430,0.000859,True


The 100-roll result does not reject the model at 5%; the 1,000-roll result does. Neither result measures the probability that the die is fair. A test based on the mean also cannot detect every unfair die: an unfair die could still have mean 3.5.


## 3. Compare strength-test evidence: `np.sqrt()` and `model.cdf()`

A manufacturer claims a mean polymer tensile strength of **50 MPa**. The lecture asks whether the mean is **below** that target:

$$H_0:\mu=50,\qquad H_A:\mu<50.$$

Each sample below has $\bar x=48$ MPa. Reproduce the lecture's **large-sample normal approximation** using the reported SD:

$$\widehat{\mathrm{SE}}=\frac{\mathrm{SD}}{\sqrt n},\qquad
z\approx\frac{\bar x-50}{\widehat{\mathrm{SE}}}.$$

Assume independent, representative measurements. The SD describes variation among specimens; the SE describes uncertainty in the mean. The reported SD is **not a known population $\sigma$**; Student's $t$ is the usual mean-test reference when $\sigma$ is estimated, as in Section 4.

Use the **signed** statistic in `cdf()` for this lower-tail question.


In [5]:
strength_mu0 = 50
strength_mean = 48
strength_n = np.array([500, 500, 100])
strength_sd = np.array([15.3, 33.6, 15.3])  # reported SDs, in MPa

strength_se = strength_sd / np.sqrt(strength_n)
strength_z = (strength_mean - strength_mu0) / strength_se
strength_p = normal_model.cdf(strength_z)  # HA: mean < 50 MPa

pd.DataFrame({
    "n": strength_n,
    "Reported SD (MPa)": strength_sd,
    "Estimated SE (MPa)": strength_se,
    "Approximate z": strength_z,
    "Lower-tail P (normal)": strength_p,
    "Reject H0 at 5%": strength_p <= alpha,
}, index=["Original", "More variable", "Smaller sample"])


,n,Reported SD (MPa),Estimated SE (MPa),Approximate z,Lower-tail P (normal),Reject H0 at 5%
Original,500,15.3,0.684237,-2.922965,0.001734,True
More variable,500,33.6,1.502638,-1.330993,0.091596,False
Smaller sample,100,15.3,1.530000,-1.307190,0.095574,False


## 4. Test a mean with unknown $\sigma$: `stats.sem()` and `stats.t()`

The instrument-calibration example uses readings **78, 83, 68, 72, and 88 ppm** from a reference gas containing **70 ppm**. Test $H_0:\mu=70$ against $H_A:\mu\ne70$.

Assume independent normal measurement errors with unknown SD. With only five readings, this assumption matters. `std(ddof=1)` uses denominator $n-1$; `stats.sem(values, ddof=1)` calculates $s/\sqrt n$.

$$t=\frac{\bar x-\mu_0}{s/\sqrt n},\qquad df=n-1,\qquad P=2\Pr(T_{df}\ge|t|).$$

Create `t_model = stats.t(df=gas_df)`, then use its `sf()` method. Use the **estimated SE**, not the sample SD, in the test-statistic denominator.


In [8]:
gas_readings = np.array([78, 83, 68, 72, 88])
gas_mu0 = 70
gas_mean = gas_readings.mean()
gas_sd = gas_readings.std(ddof=1)
gas_se = stats.sem(gas_readings, ddof=1)
print(gas_se)
print(gas_sd/np.sqrt(gas_readings.size))
gas_df = gas_readings.size - 1
gas_t = (gas_mean - gas_mu0) / gas_se

t_model = stats.t(df=gas_df)
gas_p_manual = 2 * t_model.sf(abs(gas_t))

gas_p_manual


3.6110940170535573
3.6110940170535573


0.09688798643949774

## 5. Run the test directly: `stats.ttest_1samp()`

Pass the **raw observations** to `stats.ttest_1samp()`. `popmean` is the null benchmark, not the observed sample mean. Read the result's `.statistic` and `.pvalue` attributes and compare them with the calculation above.

The `alternative` choices are `"less"`, `"greater"`, and `"two-sided"`. Keep `"two-sided"` for the stated calibration question.


In [9]:
gas_result = stats.ttest_1samp(
    gas_readings,
    popmean=gas_mu0,
    alternative="two-sided",
)

print(gas_result.statistic)
print(gas_result.pvalue)


2.1600102249246733
0.09688798643949774


### Connect to the confidence-interval notebook: `stats.t.interval()`

Reuse the same sample mean, estimated SE, and degrees of freedom. For a CI for the mean, set `loc=sample mean` and **`scale=estimated SE`**.

Subtract the 70 ppm reference from both bounds to obtain the CI for **bias**, $\mu-70$.

For these data, the 95% interval includes 70 ppm and the matching two-sided test has $P>0.05$. Failing to reject does **not** demonstrate zero bias or establish that the instrument meets an engineering tolerance.


In [10]:
gas_ci = stats.t.interval(
    confidence=1 - alpha,
    df=gas_df,
    loc=gas_mean,
    scale=gas_se,
)
gas_bias_ci = np.array(gas_ci) - gas_mu0

print(f"Estimated bias: {gas_mean - gas_mu0:.2f} ppm")
print(f"95% CI for the mean: [{gas_ci[0]:.2f}, {gas_ci[1]:.2f}] ppm")
print(f"95% CI for the bias: [{gas_bias_ci[0]:.2f}, {gas_bias_ci[1]:.2f}] ppm")
print(f"70 ppm inside the mean CI? {gas_ci[0] <= gas_mu0 <= gas_ci[1]}")
print(f"Two-sided P: {gas_result.pvalue:.6f}")
print("Decision at 5%:", "Reject H0" if gas_result.pvalue <= alpha else "Fail to reject H0")


Estimated bias: 7.80 ppm
95% CI for the mean: [67.77, 87.83] ppm
95% CI for the bias: [-2.23, 17.83] ppm
70 ppm inside the mean CI? True
Two-sided P: 0.096888
Decision at 5%: Fail to reject H0


### General Python function reference

| Purpose | General syntax | Notes |
|---|---|---|
| Estimated SE of a mean | `stats.sem(values, ddof=1)` | Returns $s/\sqrt n$. |
| Standard normal reference | `model = stats.norm()` | Creates a standard normal distribution object. |
| Student's $t$ reference | `model = stats.t(df=...)` | For a one-sample mean, use $df=n-1$. |
| Density height | `model.pdf(value)` | A curve height, not a $P$-value. |
| Lower-tail area | `model.cdf(statistic)` | Use the signed statistic for a lower-tail test. |
| Upper-tail area | `model.sf(statistic)` | Use the signed statistic for an upper-tail test. |
| Symmetric two-sided area | `2 * model.sf(abs(statistic))` | Applies to standard normal and centered $t$ references. |
| One-sample mean test | `stats.ttest_1samp(values, popmean, alternative="two-sided")` | Returns `.statistic` and `.pvalue`; alternatives also include `"less"` and `"greater"`. |
| CI using the $t$ distribution | `stats.t.interval(confidence, df, loc, scale)` | For a mean, use `loc=mean` and `scale=SE`. |

Direct calls such as `stats.norm.cdf(z)` and `stats.t.sf(t, df=...)` are equivalent to using the corresponding distribution object's methods. Use `help(function_name)` or `function_name?` in Jupyter for arguments.

**Interpretation reminder:** A $P$-value describes results at least as extreme as observed, in the direction specified by $H_A$, **assuming the null model is true**. It is not the probability that $H_0$ is true. A test does not repair dependence, biased sampling, or an incorrect probability model.


## 6. Final exercise: A new calibration check

Five new readings of the same **70 ppm** reference gas are **72, 74, 71, 75, and 73 ppm**. Assume independent normal measurement errors with unknown SD.

Use `stats.ttest_1samp()` to test $H_0:\mu=70$ against $H_A:\mu\ne70$. Then use `stats.sem()` and `stats.t.interval()` to compute the 95% CI for the mean. Report the test statistic, $P$-value, and interval. Does the test decision at $\alpha=0.05$ agree with whether 70 ppm lies inside the interval?


In [23]:
new_measurements = np.array([72, 74, 71, 75, 73])

new_mean = new_measurements.mean()
print(new_mean)
new_sd = new_measurements.std(ddof =1)

new_se = stats.sem(new_measurements)

print(new_se)

print(new_sd/np.sqrt(new_measurements.size))

new_gas_result = stats.ttest_1samp(
    new_measurements,
    popmean=gas_mu0,
    alternative="two-sided",
)
new_gas_result.pvalue

new_ci = stats.t.interval(confidence = 1-alpha, df = new_measurements.size - 1, loc = new_mean, scale = new_se)

new_ci


73.0
0.7071067811865476
0.7071067811865476


(71.03675683852244, 74.96324316147756)